# Post-Hoc Evaluation of Trained Model
This notebook performs post-hoc evaluation of a trained model using W&B. It includes validation dataset evaluation and plot generation.

## Setup and Imports

In [1]:
# Import required libraries
%load_ext autoreload
%autoreload 1
from pathlib import Path
import torch
import lightning as L

import wandb
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.config import pretty_config, find_wandb_run, get_current_config
from src.evaluation import PlotsCallback
import src.models
import src.data_loaders
from src.models import FlowModule
# Set up logging
import logging
from src.logging_util import handler

log_level = logging.INFO
logging.basicConfig(level=log_level, handlers=[handler])
logging.getLogger('src').setLevel(log_level)
logger = logging.getLogger(__name__)
logger.setLevel(log_level)

### Select run and model artifact

In [4]:
PROJECT = "flowtoy"
FIND_RUN = "2_200e_b128x30_levy"
selected_run = find_wandb_run(FIND_RUN)
assert isinstance(selected_run, wandb.apis.public.Run), f"Run {FIND_RUN} not found" # type: ignore


 ✅ Run ID: y2gn0fqz, Run Name: 2_200e_b128x30_levy
 Created at: 2025-04-21T15:37:30Z
   Run URL: https://wandb.ai/tresoor/flowtoy/runs/y2gn0fqz


In [5]:
def find_and_download_model(run):
    artifacts = [a for a in run.logged_artifacts() if a.type == "model"]
    for artifact in artifacts:
        print(
        f"{artifact.name}\n  > Type: {artifact.type}, Version: {artifact.version}, aliases: {artifact.aliases}, size: {artifact.size:_}, updated: {artifact.updated_at}, description: {artifact.description}"
        )

    if len(artifacts) == 0:
        print("No model artifacts found")
        raise ValueError("No model artifacts found")
    elif len(artifacts) == 1:
        artifact = artifacts[0]
        print(f"Single model artifact found, so using it.")
    else:
        artifact_name = input(f"Enter the name of the artifact to use (default: {artifacts[0].name}): ")
        if artifact_name == "":
            artifact_name = artifacts[0].name
        artifact = wandb.Api().artifact(artifact_name)
        print(f"Using artifact {artifact.name}")
    print("Downloading artifact...")
    artifact_dir = artifact.download()
    print("Stored model locally in", artifact_dir)
    # Log model summary
    # logger.info("Model loaded. Summary:")
    # model.log_summary(C)
    # Get the number of steps the model has trained

    # load checkpoint
    return Path(artifact_dir) / "model.ckpt"


checkpoint_path = find_and_download_model(selected_run)

2025-04-21-2_200e_b128x30_levy:v15
  > Type: model, Version: v15, aliases: ['latest', 'best'], size: 936_823_375, updated: 2025-04-21T16:18:50Z, description: None
Single model artifact found, so using it.


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Downloading large artifact 2025-04-21-2_200e_b128x30_levy:v15, 893.42MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:2:25.4


Stored model locally in /Users/milan/Code/fusion/experiments/artifacts/2025-04-21-2_200e_b128x30_levy:v15


In [6]:
# Load the configuration file
config = selected_run.config
print(pretty_config(config))

{'Attn': [True, False, False, False],
 'Cols': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'],
 'Crop': 2000,
 'data': {'dir': './data/',
          'cols': {'c': ['IP', 'gas_fringes', 'NBI', 'ECRH', 'a_minor', 'KAPPA', 'DELTA'],
                   'x': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'],
                   'meta': ['ShotNum', 'time'],
                   'label': 'LHD_label'},
          'file': '2024_05_01-NaNsFiltered.parquet',
          'Class': 'ShotFlowDS',
          'seq_length': 256,
          'test_shots': [53623, 57013, 57094, 57732, 60813, 60814, 61028, 61237, 63306, 63878, 64365, 64386, 64393, 64678, 64686, 64770, 64857, 65481, 67112,
                         68631, 68697, 69514, 73368, 73631, 73935, 75264, 76304, 76702, 77089, 77193, 77196, 77409, 77595, 77598, 77599, 77602, 77604],
          'crop_margin': 2000,
          'pre_shuffle': True,
          'sample_rate': 10000,
          'train_shots': [26386, 29511, 30043, 30197, 30225, 30262, 30268, 30290, 30310, 31211, 

## Load Model and Configuration from W&B
Use the W&B API to load the trained model and its configuration. Initialize the model using the configuration parameters.

In [7]:
# Load configuration and initialize W&B

eval_run = wandb.init(
    project=PROJECT,
    name=selected_run.name,
    id=selected_run.id,
    resume="must",
    mode="disabled",
    config=config,
)

# Load model configuration
C = get_current_config()
print(pretty_config(C))
assert C == config, "Config from wandb run and config from artifact do not match"

ModelClass = getattr(src.models, C.model.Class)
assert issubclass(ModelClass, FlowModule), "ModelClass must be a subclass of FlowModule"

checkpoint_local_file = Path(artifact_dir) / "model.ckpt"
model = FlowModule.load_from_checkpoint(checkpoint_local_file)
# Log model summary
# logger.info("Model loaded. Summary:")
# model.log_summary(C)
# Get the number of steps the model has trained

# load checkpoint
epoch = model.current_epoch
logger.info("Model has trained for %d epochs", epoch)

{'Attn': [True, False, False, False], 'Cols': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'], 'Crop': 2000, 'data': {'dir': './data/', 'cols': {'c': ['IP', 'gas_fringes', 'NBI', 'ECRH', 'a_minor', 'KAPPA', 'DELTA'], 'x': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'], 'meta': ['ShotNum', 'time'], 'label': 'LHD_label'}, 'file': '2024_05_01-NaNsFiltered.parquet', 'Class': 'ShotFlowDS', 'seq_length': 256, 'test_shots': [53623, 57013, 57094, 57732, 60813, 60814, 61028, 61237, 63306, 63878, 64365, 64386, 64393, 64678, 64686, 64770, 64857, 65481, 67112, 68631, 68697, 69514, 73368, 73631, 73935, 75264, 76304, 76702, 77089, 77193, 77196, 77409, 77595, 77598, 77599, 77602, 77604], 'crop_margin': 2000, 'pre_shuffle': True, 'sample_rate': 10000, 'train_shots': [26386, 29511, 30043, 30197, 30225, 30262, 30268, 30290, 30310, 31211, 31554, 31650, 31718, 31807, 31839, 32191, 32195, 32592, 32716, 32911, 33188, 33271, 33281, 33459, 33567, 33942, 34010, 34309, 42197, 42514, 43454, 45103, 45105, 46853, 47962, 

NameError: name 'artifact_dir' is not defined

In [ ]:
trainer = L.Trainer(
    accelerator="auto",
    devices=1,
    max_epochs=1,
    fast_dev_run=True,
)
# print(trainer)
trainer.fit(model, ckpt_path=checkpoint_local_file)


You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
/Users/milan/.local/share/virtualenvs/experiments-2XM9RiFo/lib/python3.11/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: PossibleUserWarning:

You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

Restoring states from the checkpoint path at /Users/milan/Code/fusion/experiments/artifacts/2025-04-21-Prof2_art_delete_e10_levy:v5/model.ckpt
/Users/milan/.local/share/virtualenvs/experiments-2XM9RiFo/lib/python3.11/site-packages/lightning/pytorch/trainer/call.py:282: UserWarning:

Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided durin

MisconfigurationException: You restored a checkpoint with current_epoch=9, but you have set Trainer(max_epochs=1).

## Load Validation Dataset
Load the validation dataset using the same DataSetClass and parameters as in `run.py`.

In [ ]:
# Load validation dataset
DataSetClass = getattr(src.data_loaders, C.data.Class)
val_set = DataSetClass(**C.data, train=False)

# Create DataLoader for validation dataset
val_loader = DataLoader(
    val_set,
    batch_size=1,
    shuffle=False,
)

logger.info("Validation dataset loaded.")

15:12:56 (+45.85s) src.data_loaders[INFO]:Using 37 shots from ./data/2024_05_01-NaNsFiltered.parquet
15:12:56 (+ 0.15s) src.data_loaders[INFO]:Column 'time_step      ': min=0.0000      max=26138.0000  mean=8040.4892   std=5206.8242   nans=0         
15:12:56 (+ 0.00s) src.data_loaders[INFO]:Column 'ShotNum        ': min=53623.0000  max=77604.0000  mean=69145.2513  std=7638.5923   nans=0         
15:12:56 (+ 0.01s) src.data_loaders[INFO]:Column 'IP             ': min=0.0000      max=1.0000      mean=0.4399      std=0.1395      nans=0         
15:12:56 (+ 0.01s) src.data_loaders[INFO]:Column 'gas_fringes    ': min=0.0000      max=1.0000      mean=0.3031      std=0.1447      nans=0         
15:12:56 (+ 0.01s) src.data_loaders[INFO]:Column 'NBI            ': min=0.0000      max=1.0000      mean=0.3054      std=0.3801      nans=0         
15:12:56 (+ 0.01s) src.data_loaders[INFO]:Column 'ECRH           ': min=0.0000      max=1.0000      mean=0.0649      std=0.1680      nans=0         
15:12

## Run Evaluation and Generate Plots
Run the evaluation functions on the validation dataset and generate plots using the plot functions defined in `evaluation.py`.

In [ ]:
# Run evaluation
# N_STEPS = C.evaluation.n_steps
N_STEPS = 50
logger.info("Starting evaluation using %s steps...", N_STEPS)
batch = next(iter(val_loader))
evaluation_output = model.evaluate(batch, n_steps=N_STEPS)
logger.info("Evaluation completed.")



15:12:58 (+ 1.24s) __main__[INFO]:Starting evaluation using 50 steps...
Integrating path: 100%|██████████| 49/49 [00:08<00:00,  5.66it/s]
15:13:07 (+ 8.87s) __main__[INFO]:Evaluation completed.


In [ ]:
from src.plotters.plot_animations import animated_trajectory_plotly

animated_trajectory_plotly(**evaluation_output, title_base="Animation")

In [37]:
animated_trajectory_plotly(**evaluation_output, title_base="Animation")